In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==============================================================================
# 🎛️ PARAMETER DIREKTORI UTAMA
# ==============================================================================
BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/rep_code"
MODEL_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20'
EMB_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909'

# Path merujuk ke file HDF5 yang baru saja berhasil Anda buat
PATH_HDF5 = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/hdf5_output/dataset_indonesia.hdf5'
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'

PATH_REPORT_1C = os.path.join(PATH_DEMO_DIR, 'mcu_quake_indonesia_1C_HDF5_report.csv')
PATH_REPORT_3C = os.path.join(PATH_DEMO_DIR, 'mcu_quake_indonesia_3C_HDF5_report.csv')

if PATH_DEMO_DIR not in sys.path: sys.path.append(PATH_DEMO_DIR)
if BASE_REP not in sys.path: sys.path.append(BASE_REP)

try:
    from Library import utils, dataset
except ImportError:
    print("❌ Gagal memuat Library.")
    sys.exit()

# ==============================================================================
# 🔥 FUNGSI WORKER: INFERENSI AI DARI MATRIKS BERSIH
# ==============================================================================
def inferensi_satu_event(eid, matriks_le, matriks_no, embedding_model, pdf_1C, pdf_3C):
    try:
        # Pecah Matriks 3C menjadi Z, N, E
        le_z, le_n, le_e = matriks_le[0], matriks_le[1], matriks_le[2]
        no_z, no_n, no_e = matriks_no[0], matriks_no[1], matriks_no[2]
        
        # --- EKSEKUSI 1C-Z ---
        emb_le_1c = utils.latent_codes_1D(le_z, embedding_model).reshape(1, -1)
        p_le_1c, _, _ = utils.infer_1C_PDFs(emb_le_1c, pdf_1C, "Kernel")
        keputusan_le_1c = 1 if p_le_1c == 0 else 0
        
        emb_no_1c = utils.latent_codes_1D(no_z, embedding_model).reshape(1, -1)
        p_no_1c, _, _ = utils.infer_1C_PDFs(emb_no_1c, pdf_1C, "Kernel")
        keputusan_no_1c = 1 if p_no_1c == 0 else 0
        
        # --- EKSEKUSI 3C ---
        emb_le_3c = np.array([
            utils.latent_codes_1D(le_z, embedding_model),
            utils.latent_codes_1D(le_n, embedding_model),
            utils.latent_codes_1D(le_e, embedding_model)
        ]).reshape(1, -1)
        p_le_3c, _, _ = utils.infer_3C_PDFs(emb_le_3c, pdf_3C, "Kernel")
        keputusan_le_3c = 1 if p_le_3c == 0 else 0
        
        emb_no_3c = np.array([
            utils.latent_codes_1D(no_z, embedding_model),
            utils.latent_codes_1D(no_n, embedding_model),
            utils.latent_codes_1D(no_e, embedding_model)
        ]).reshape(1, -1)
        p_no_3c, _, _ = utils.infer_3C_PDFs(emb_no_3c, pdf_3C, "Kernel")
        keputusan_no_3c = 1 if p_no_3c == 0 else 0
        
        return {
            'rows_1c': [
                {'File': eid, 'Type': 'LE_1C', 'Decision': keputusan_le_1c},
                {'File': eid, 'Type': 'NO_1C', 'Decision': keputusan_no_1c}
            ],
            'rows_3c': [
                {'File': eid, 'Type': 'LE_3C', 'Decision': keputusan_le_3c},
                {'File': eid, 'Type': 'NO_3C', 'Decision': keputusan_no_3c}
            ]
        }
    except Exception:
        return None

# ==============================================================================
# MODUL VISUALISASI GAYA JURNAL
# ==============================================================================
def plot_paper_style_cm(cm, title, output_name):
    TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    TPR_NO = (TN / (TN + FP)) * 100 if (TN + FP) > 0 else 0.0
    TPR_LE = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
    PPV_NO = (TN / (TN + FN)) * 100 if (TN + FN) > 0 else 0.0
    PPV_LE = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0

    fig, ax = plt.subplots(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['', ''], yticklabels=['NO', 'LE'], annot_kws={"size": 15}, ax=ax)
    ax.set_title(title, fontsize=14, pad=30, loc='center')
    ax.set_ylabel('True label', fontsize=13)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    ax.set_xlabel('Predicted', fontsize=13, labelpad=5)
    plt.yticks(fontsize=13, rotation=0)
    ax.tick_params(axis='both', which='both', length=0)

    ax.text(2.1, -0.1, 'TPR:', fontsize=13, ha='left', va='center')
    ax.text(2.1, 0.5, f'{TPR_NO:.2f}', fontsize=13, ha='left', va='center')
    ax.text(2.1, 1.5, f'{TPR_LE:.2f}', fontsize=13, ha='left', va='center')
    ax.text(-0.2, 2.2, 'PPV:', fontsize=13, ha='right', va='center')
    ax.text(0.5, 2.2, f'{PPV_NO:.2f}', fontsize=13, ha='center', va='center')
    ax.text(1.5, 2.2, f'{PPV_LE:.2f}', fontsize=13, ha='center', va='center')

    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.5)

    plt.subplots_adjust(right=0.75, bottom=0.2)
    plt.savefig(os.path.join(PATH_DEMO_DIR, f'paper_style_cm_{output_name}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_bar_metrics_paper_style(cm, title, output_name):
    TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    acc = ((TP + TN) / (TP + TN + FP + FN)) * 100
    rec = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
    prec = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.bar(['Accuracy\n(Global)', 'Recall\n(True Positive Rate)', 'Precision\n(Positive Predictive)', 'F1-Score\n(Harmonic Mean)'], [acc, rec, prec, f1], color='#c93b4a', edgecolor='black', linewidth=1.5, width=0.55)
    ax.set_ylim(0, 115)
    ax.yaxis.grid(True, linestyle='--', color='gray', alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_ylabel('Performance Score (%)', fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=10)
    ax.tick_params(axis='x', labelsize=10.5)

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.2f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_DEMO_DIR, f'bar_chart_metrics_{output_name}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def visualisasi_hasil_riset_hdf5(report_path, hdf5_path, mode="1C_Z"):
    df_report = pd.read_csv(report_path)
    
    y_true = df_report['Type'].apply(lambda x: 1 if 'LE' in x else 0).tolist()
    y_pred = df_report['Decision'].tolist()
    cm = confusion_matrix(y_true, y_pred)
    
    plot_bar_metrics_paper_style(cm, f"Classification Metrics: {mode} (HDF5 Blind Test)", mode)
    plot_paper_style_cm(cm, f"MCU-Quake {mode} (Blind Test)", mode)

    # Ekstrak magnitudo langsung dari HDF5
    mag_data = []
    with h5py.File(hdf5_path, 'r') as hdf:
        grp_le = hdf['earthquake']
        for _, row in df_report[df_report['Type'].str.contains('LE')].iterrows():
            eid = str(row['File']).strip()
            if eid in grp_le:
                # Menarik Atribut Metadata (Magnitudo)
                mag = grp_le[eid].attrs.get('mag', np.nan)
                if not pd.isna(mag):
                    status = 'Detected (TP)' if row['Decision'] == 1 else 'Missed (FN)'
                    mag_data.append({'magnitude': float(mag), 'Status': status})
                    
    df_gempa = pd.DataFrame(mag_data)
    
    if not df_gempa.empty:
        plt.figure(figsize=(10, 6))
        sns.kdeplot(data=df_gempa, x='magnitude', hue='Status', fill=True, common_norm=False, palette='viridis')
        plt.title(f'Distribution of Magnitude: Impact of Blind Temporal Window ({mode})')
        plt.xlabel('Earthquake Magnitude')
        plt.ylabel('Density')
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(PATH_DEMO_DIR, f'magnitude_kde_{mode}.png'), dpi=300)
        plt.close()

# ==============================================================================
# 🚀 CORE PROCESS: MEMBACA HDF5 DENGAN MANAJEMEN ANTRIAN RAM
# ==============================================================================
def jalankan_stress_test_kombinasi_hdf5():
    print("\n" + "="*90)
    print("🚀 STARTING: HDF5 INFERENCE (1C & 3C BLIND TEST BERSAMAAN)")
    print("="*90)
    
    embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    print("⏳ Memuat Typical Embeddings (Z, N, E)...")
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
    embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
    
    pdf_1C = utils.embedding_PDFs_1D(embedding_Z)
    pdf_3C = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)

    report_rows_1c, report_rows_3c = [], []
    
    print(f"📂 Membuka brankas HDF5...")
    with h5py.File(PATH_HDF5, 'r') as hdf:
        grp_le = hdf['earthquake']
        grp_no = hdf['noise']
        semua_eid = list(grp_le.keys())
        
        # Sesuaikan dengan Core CPU Mac M3 Pro
        MAX_THREADS = min(32, (os.cpu_count() or 1) + 4)
        
        print(f"⚡ Menyiapkan {MAX_THREADS} Pekerja. Mulai memompa data ke memori...")
        with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
            futures = []
            
            # 🔥 LOOP PERTAMA: Progress bar untuk memuat data dari SSD ke RAM
            for eid in tqdm(semua_eid, desc="1/2: Memuat Data & Antrean"):
                mat_le = grp_le[eid][:]  # Pindahkan matriks gempa ke RAM
                mat_no = grp_no[eid][:]  # Pindahkan matriks noise ke RAM
                futures.append(executor.submit(inferensi_satu_event, eid, mat_le, mat_no, embedding_model, pdf_1C, pdf_3C))
                
            # 🔥 LOOP KEDUA: Progress bar untuk Inferensi AI paralel
            for future in tqdm(as_completed(futures), total=len(futures), desc="2/2: Eksekusi AI (1C & 3C)"):
                hasil = future.result()
                if hasil is not None:
                    report_rows_1c.extend(hasil['rows_1c'])
                    report_rows_3c.extend(hasil['rows_3c'])

    print("\n📊" + "="*34 + " MENYIMPAN LAPORAN & VISUALISASI " + "="*34)
    pd.DataFrame(report_rows_1c).to_csv(PATH_REPORT_1C, index=False)
    pd.DataFrame(report_rows_3c).to_csv(PATH_REPORT_3C, index=False)
    
    print("🎨 Render Visualisasi 1C HDF5...")
    visualisasi_hasil_riset_hdf5(PATH_REPORT_1C, PATH_HDF5, mode="1C_Z")
    
    print("🎨 Render Visualisasi 3C HDF5...")
    visualisasi_hasil_riset_hdf5(PATH_REPORT_3C, PATH_HDF5, mode="3C")
    
    print(f"\n✅ Selesai! Semua laporan dan grafik tersimpan di:\n📂 {PATH_DEMO_DIR}")

if __name__ == "__main__":
    jalankan_stress_test_kombinasi_hdf5()